# FaceInsight-AI EDA

# Overview
This notebook performs Exploratory Data Analysis (EDA) on the CelebA dataset using KaggleHub. It examines three CSV files extracted from the dataset:
- `list_attr_celeba.csv` â€” facial attribute annotations
- `list_bbox_celeba.csv` â€” bounding box coordinates for face regions
- `list_eval_partition.csv` â€” train/validation/test split indicators

The notebook covers data loading, summary statistics, and visualization of distributions, correlations, and scatter relationships across each dataset.

# Setup
Import required libraries and set up the environment for analysis.

In [3]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# The original error occurred because file_path was an empty string.
# We need to specify a file name from the dataset with its extension.
# For example, 'list_attr_celeba.csv' is a file within the 'jessicali9530/celeba-dataset'.
file_path = "list_attr_celeba.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "jessicali9530/celeba-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

<>:10: SyntaxWarning: invalid escape sequence '\l'
<>:10: SyntaxWarning: invalid escape sequence '\l'
C:\Users\intel\AppData\Local\Temp\ipykernel_22108\4194084346.py:10: SyntaxWarning: invalid escape sequence '\l'
  file_path = "archive\list_attr_celeba.csv"
C:\Users\intel\AppData\Local\Temp\ipykernel_22108\4194084346.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
C:\Users\intel\AppData\Local\Temp\ipykernel_22108\4194084346.py:10: SyntaxWarning: invalid escape sequence '\l'
  file_path = "archive\list_attr_celeba.csv"


KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://kaggle.com/datasets/jessicali9530/celeba-dataset/versions/2
Please make sure you specified the correct resource identifiers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

# Utility Functions
Helper functions for visualizing data distributions, correlation matrices, and scatter plots.

In [ ]:
def plotPerColumnDistribution(df, nGraphPerRow, nGraphShown):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]]
    if df.shape[1] == 0:
        print('No valid columns to plot')
        return
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int(np.ceil(nCol / nGraphPerRow))
    plt.figure(figsize=(nGraphPerRow * 6, nGraphRow * 4))
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        col = columnNames[i]
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col].hist(bins=30)
        else:
            df[col].value_counts().plot(kind='bar')
        plt.title(f'{col} Distribution')
        plt.xlabel(col)
    plt.tight_layout()
    plt.show()

def plotCorrelationMatrix(df, size):
    corr = df.select_dtypes(include=[np.number]).corr()
    if corr.shape[0] < 2:
        print('Not enough numeric columns for correlation matrix')
        return
    plt.figure(figsize=(size, size))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Matrix')
    plt.show()

def plotScatterMatrix(df, plotSize, textSize):
    numeric = df.select_dtypes(include=[np.number])
    if numeric.shape[1] < 2:
        print('Not enough numeric columns for scatter matrix')
        return
    if numeric.shape[1] > 10:
        numeric = numeric.iloc[:, :10]
    plt.figure(figsize=(plotSize, plotSize))
    pd.plotting.scatter_matrix(numeric, figsize=(plotSize, plotSize))
    plt.show()

# Data Download
Download the CelebA dataset from Kaggle using `kagglehub`. The dataset is persisted locally and can be reused across notebook sessions.

In [ ]:
import kagglehub
import os

# Download the entire dataset to a local directory
# kagglehub.dataset_download returns the path where the dataset is downloaded
# We can then use this path as our DATA_DIR for local file operations.
DATA_DIR = kagglehub.dataset_download("jessicali9530/celeba-dataset")

print(f"Dataset downloaded to: {DATA_DIR}")
print(f"Contents of {DATA_DIR}:")
print(os.listdir(DATA_DIR))

# DataFrame 1 â€” Facial Attributes (`list_attr_celeba.csv`)
This dataset contains 40 binary facial attributes (e.g., Smiling, Male, Wearing_Hat) annotated for each celebrity image. We load a sample of 1000 rows and visualize the distributions and correlations.

In [ ]:
nRowsRead = 1000
df1 = pd.read_csv(os.path.join(DATA_DIR, 'list_attr_celeba.csv'), delimiter=',', nrows=nRowsRead)
df1.dataframeName = 'list_attr_celeba.csv'
nRow, nCol = df1.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df1.head(5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plotPerColumnDistribution(df, nGraphPerRow, nGraphShown):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]]
    if df.shape[1] == 0:
        print('No valid columns to plot')
        return
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int(np.ceil(nCol / nGraphPerRow))
    plt.figure(figsize=(nGraphPerRow * 6, nGraphRow * 4))
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        col = columnNames[i]
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col].hist(bins=30)
        else:
            df[col].value_counts().plot(kind='bar')
        plt.title(f'{col} Distribution')
        plt.xlabel(col)
    plt.tight_layout()
    plt.show()

plotPerColumnDistribution(df1, 10, 5)

In [ ]:
def plotCorrelationMatrix(df, size):
    corr = df.select_dtypes(include=[np.number]).corr()
    if corr.shape[0] < 2:
        print('Not enough numeric columns for correlation matrix')
        return
    plt.figure(figsize=(size, size))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Matrix')
    plt.show()

plotCorrelationMatrix(df1, 10)

In [ ]:
plotScatterMatrix(df1, 20, 10)

# DataFrame 2 â€” Bounding Boxes (`list_bbox_celeba.csv`)
This dataset contains bounding box coordinates (`x`, `y`, `width`, `height`) for face regions in each image, along with `confidence` and `target`. We visualize the distributions and correlations.

In [ ]:
nRowsRead = 1000
df2 = pd.read_csv(os.path.join(DATA_DIR, 'list_bbox_celeba.csv'), delimiter=',', nrows=nRowsRead)
df2.dataframeName = 'list_bbox_celeba.csv'
nRow, nCol = df2.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df2.head(5)

In [ ]:
plotPerColumnDistribution(df2, 10, 5)

In [ ]:
plotCorrelationMatrix(df2, 8)

In [ ]:
plotScatterMatrix(df2, 12, 10)

# DataFrame 3 â€” Evaluation Partition (`list_eval_partition.csv`)
This dataset assigns each image to a partition (`0` = Training, `1` = Validation, `2` = Test), enabling reproducibility of train/val/test splits.

In [ ]:
nRowsRead = 1000
df3 = pd.read_csv(os.path.join(DATA_DIR, 'list_eval_partition.csv'), delimiter=',', nrows=nRowsRead)
df3.dataframeName = 'list_eval_partition.csv'
nRow, nCol = df3.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df3.head(5)

In [ ]:
plotPerColumnDistribution(df3, 10, 5)

In [ ]:
partition = pd.read_csv(os.path.join(DATA_DIR, 'list_eval_partition.csv'))
attrs = pd.read_csv(os.path.join(DATA_DIR, 'list_attr_celeba.csv'))
df = attrs.merge(partition, on='image_id')

train_df = df[df.partition == 0]
val_df   = df[df.partition == 1]
test_df  = df[df.partition == 2]

In [ ]:
IMG_SIZE = 512  # start small, bump to 224 later if it helps

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, label

def make_ds(df, label_col, shuffle=False):
    paths = (img_dir + df.image_id).values
    labels = (df[label_col] == 1).astype(int).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
attrs = pd.read_csv(os.path.join(DATA_DIR, 'list_attr_celeba.csv'))
bbox = pd.read_csv(os.path.join(DATA_DIR, 'list_bbox_celeba.csv'))
landmarks = pd.read_csv(os.path.join(DATA_DIR, 'list_landmarks_align_celeba.csv'))
partition = pd.read_csv(os.path.join(DATA_DIR, 'list_eval_partition.csv'))

df = attrs.merge(bbox, on='image_id') \
          .merge(landmarks, on='image_id') \
          .merge(partition, on='image_id')

attr_cols = [c for c in attrs.columns if c != 'image_id']
bbox_cols = ['x_1', 'y_1', 'width', 'height']
landmark_cols = [c for c in landmarks.columns if c != 'image_id']

print(df.shape)
df.head()

In [ ]:
img_dir = os.path.join(DATA_DIR, 'img_align_celeba', 'img_align_celeba') + os.sep

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, row) in zip(axes, df.sample(4, random_state=1).iterrows()):
    img = plt.imread(img_dir + row['image_id'])
    ax.imshow(img)
    ax.add_patch(plt.Rectangle((row.x_1, row.y_1), row.width, row.height,
                                edgecolor='lime', facecolor='none', linewidth=2))
    xs = [row[c] for c in landmark_cols if c.endswith('_x')]
    ys = [row[c] for c in landmark_cols if c.endswith('_y')]
    ax.scatter(xs, ys, c='red', s=20)
    ax.axis('off')
plt.show()

In [ ]:
IMG_SIZE = 128

train_df = df[df.partition == 0].reset_index(drop=True)
val_df   = df[df.partition == 1].reset_index(drop=True)
test_df  = df[df.partition == 2].reset_index(drop=True)

def normalize_targets(d):
    d = d.copy()
    d[bbox_cols[0]] = d[bbox_cols[0]] / 178.0   # x_1 / img width
    d[bbox_cols[2]] = d[bbox_cols[2]] / 178.0   # width
    d[bbox_cols[1]] = d[bbox_cols[1]] / 218.0   # y_1 / img height
    d[bbox_cols[3]] = d[bbox_cols[3]] / 218.0   # height
    for c in landmark_cols:
        d[c] = d[c] / (178.0 if c.endswith('_x') else 218.0)
    return d

train_df, val_df, test_df = normalize_targets(train_df), normalize_targets(val_df), normalize_targets(test_df)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
def load_and_preprocess(path, attr, land, box):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, {'attributes': attr, 'landmarks': land, 'bbox': box}

def make_dataset(d, shuffle=False, batch_size=64):
    paths = (img_dir + d['image_id']).values
    attr = ((d[attr_cols].values == 1).astype('float32'))
    land = d[landmark_cols].values.astype('float32')
    box = d[bbox_cols].values.astype('float32')

    ds = tf.data.Dataset.from_tensor_slices((paths, attr, land, box))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

In [ ]:
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

backbone = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
backbone.trainable = False

x = backbone(inputs)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

attributes_out = Dense(len(attr_cols), activation='sigmoid', name='attributes')(x)
landmarks_out = Dense(len(landmark_cols), activation='sigmoid', name='landmarks')(x)
bbox_out = Dense(len(bbox_cols), activation='sigmoid', name='bbox')(x)

model = Model(inputs=inputs, outputs=[attributes_out, landmarks_out, bbox_out])
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss={
        'attributes': 'binary_crossentropy',
        'landmarks': 'mse',
        'bbox': 'mse'
    },
    loss_weights={
        'attributes': 1.0,
        'landmarks': 5.0,
      'bbox': 5.0
    },
    metrics={
        'attributes': 'accuracy',
        'landmarks': 'mae',
        'bbox': 'mae'
    }
)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
backbone.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={'attributes': 'binary_crossentropy', 'landmarks': 'mse', 'bbox': 'mse'},
    loss_weights={'attributes': 1.0, 'landmarks': 5.0, 'bbox': 5.0},
    metrics={'attributes': 'accuracy', 'landmarks': 'mae', 'bbox': 'mae'}
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
from sklearn.metrics import f1_score, average_precision_score

results = model.evaluate(test_ds, return_dict=True)
print(results)

# per-attribute F1 / mAP
attr_pred, land_pred, box_pred = model.predict(test_ds)
attr_true = (test_df[attr_cols].values == 1).astype(int)
attr_pred_labels = (attr_pred > 0.5).astype(int)

macro_f1 = f1_score(attr_true, attr_pred_labels, average='macro')
mAP = average_precision_score(attr_true, attr_pred, average='macro')
print(f"Attributes macro-F1: {macro_f1:.4f}")
print(f"Attributes mAP:      {mAP:.4f}")

land_true = test_df[landmark_cols].values
landmark_mae = np.abs(land_pred - land_true).mean()
print(f"Landmark MAE (normalized): {landmark_mae:.4f}")

box_true = test_df[bbox_cols].values
bbox_mae = np.abs(box_pred - box_true).mean()
print(f"Bbox MAE (normalized): {bbox_mae:.4f}")

In [ ]:
model.save("multitask_face_model.keras")